## QAT Test-Set Inference using PyTorch (Multi-Seed)

Evaluate QAT checkpoints on the **test set** (ImageNet val split).

- **INT8**: `checkpoints/qat/int8_in{8,4,2,1}b/seed_{1,2,42}/`
- **INT4**: `checkpoints/qat/int4_in{8,4,2,1}b/seed_{1,2,42}/`

Only runs for which training has completed (checkpoint exists) are evaluated.

Results saved to `resultsv2/test_runs/`.

In [1]:
# ── Config ────────────────────────────────────────────────────────
SPLIT = "val"
TEST_IMAGENET = "/home/pf4636/imagenet2"
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/test/qat"
SKIP_EXISTING = True

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
QAT_MOD = PYFILES / "qat_modelopt"
for p in [str(PYFILES), str(QAT_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [3]:
import json
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import modelopt.torch.opt as mto

from dataclasses import replace
from torch.utils.data import DataLoader

from src.config import ExperimentConfig, set_seed
from src.data import build_imagenet_dataset
from src.eval import evaluate
from quantize import get_model, restore_modelopt_state

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [4]:
QAT_CKPT_ROOT = PROJECT_ROOT / ".checkpoints" / "qat"
FP32_CKPT_ROOT = PROJECT_ROOT / ".checkpoints"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

QAT_CONFIGS = [
    {"qat_precision": "int8", "input_bits": [8, 4, 2, 1]},
    {"qat_precision": "int4", "input_bits": [8, 4, 2, 1]},
]

checkpoints = {}
skipped = []
for qcfg in QAT_CONFIGS:
    prec = qcfg["qat_precision"]
    for bits in qcfg["input_bits"]:
        run_name = f"{prec}_in{bits}b"
        run_dir = QAT_CKPT_ROOT / run_name
        if not run_dir.exists():
            skipped.append(f"{prec}/in{bits}b (no dir)")
            continue
        for seed_dir in sorted(run_dir.iterdir()):
            if not seed_dir.is_dir() or not seed_dir.name.startswith("seed_"):
                continue
            weights = seed_dir / "qat_modelopt_best.pth"
            mostate = seed_dir / "qat_modelopt_best_mostate.pt"
            seed_num = int(seed_dir.name.split("_")[1])
            fp32_ckpt = FP32_CKPT_ROOT / f"fp32_{bits}bit" / seed_dir.name / "best.pth"
            if not weights.exists() or not mostate.exists() or not fp32_ckpt.exists():
                skipped.append(f"{prec}/in{bits}b/{seed_dir.name}")
                continue
            checkpoints[(prec, bits, seed_dir.name)] = {
                "weights": weights,
                "mostate": mostate,
                "fp32_ckpt": fp32_ckpt,
                "seed": seed_num,
            }

print(f"QAT checkpoints found: {len(checkpoints)}")
for (prec, bits, seed), paths in checkpoints.items():
    print(f"  {prec} / in{bits}b / {seed}")
if skipped:
    print(f"\nSkipped (not complete): {', '.join(skipped)}")
print(f"Test set: {TEST_IMAGENET}")

QAT checkpoints found: 24
  int8 / in8b / seed_1
  int8 / in8b / seed_2
  int8 / in8b / seed_42
  int8 / in4b / seed_1
  int8 / in4b / seed_2
  int8 / in4b / seed_42
  int8 / in2b / seed_1
  int8 / in2b / seed_2
  int8 / in2b / seed_42
  int8 / in1b / seed_1
  int8 / in1b / seed_2
  int8 / in1b / seed_42
  int4 / in8b / seed_1
  int4 / in8b / seed_2
  int4 / in8b / seed_42
  int4 / in4b / seed_1
  int4 / in4b / seed_2
  int4 / in4b / seed_42
  int4 / in2b / seed_1
  int4 / in2b / seed_2
  int4 / in2b / seed_42
  int4 / in1b / seed_1
  int4 / in1b / seed_2
  int4 / in1b / seed_42
Test set: /home/pf4636/imagenet2


In [5]:
def load_qat_model(prec: str, bits: int, seed_name: str) -> nn.Module:
    paths = checkpoints[(prec, bits, seed_name)]
    model = get_model(str(paths["fp32_ckpt"]), num_classes=100)
    restore_modelopt_state(model, str(paths["mostate"]))
    ckpt = torch.load(paths["weights"], map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt
    model.load_state_dict(state)
    model = model.to(DEVICE)
    model.eval()
    return model

def build_test_loader(cfg: ExperimentConfig) -> DataLoader:
    test_cfg = replace(cfg, imagenet_path=TEST_IMAGENET)
    dataset = build_imagenet_dataset(test_cfg, split=SPLIT)
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
    )

## QAT Evaluation (All Seeds)

In [6]:
OUT_DIR = Path(OUTPUT_ROOT).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

all_records = []
criterion = nn.CrossEntropyLoss()

for (prec, bits, seed_name), paths in checkpoints.items():
    seed_num = paths["seed"]
    run_id = f"torch_qat_{prec}_b{bits}_cuda"
    result_path = OUT_DIR / seed_name / run_id / "result.json"

    if SKIP_EXISTING and result_path.exists():
        print(f"  {prec}/in{bits}b/{seed_name}: exists, skipping")
        with open(result_path) as f:
            all_records.append(json.load(f))
        continue

    print(f"\n--- QAT {prec.upper()}, input_bits={bits}, {seed_name} ---")
    cfg = ExperimentConfig(
        backend="pytorch",
        device="cuda",
        batch_size=1,
        seed=seed_num,
        num_eval_batches=None,
        input_quant_bits=bits,
        model_precision="fp32",
    )
    set_seed(cfg)

    model = load_qat_model(prec, bits, seed_name)
    test_loader = build_test_loader(cfg)

    t0 = time.perf_counter()
    tracker = evaluate(model, test_loader, cfg, criterion=criterion)
    elapsed = time.perf_counter() - t0

    r = tracker.summary()
    print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "config_extra": {
            "qat_precision": prec,
            "input_quant_bits": bits,
            "seed": seed_num,
        },
        "results": r,
        "error": None,
        "total_eval_time_sec": round(elapsed, 3),
    }

    result_path.parent.mkdir(parents=True, exist_ok=True)
    with open(result_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    all_records.append(payload)
    del model
    torch.cuda.empty_cache()

print(f"\n{len(all_records)} runs complete.")


--- QAT INT8, input_bits=8, seed_1 ---
Inserted 107 quantizers


/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of maxpool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(
/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


[data] Filtered to 5000 samples across 100 classes.
Evaluating on 5000 batches (first 30 are warmup)...
Loading extension modelopt_cuda_ext...
Loaded extension modelopt_cuda_ext in 0.4 seconds
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/5000] Top-1: 90.00% | Top-5: 100.00% | Infer: 8.50 ms/batch
  Batch [50/5000] Top-1: 95.00% | Top-5: 100.00% | Infer: 9.76 ms/batch
  Batch [60/5000] Top-1: 93.33% | Top-5: 100.00% | Infer: 9.49 ms/batch
  Batch [70/5000] Top-1: 90.00% | Top-5: 100.00% | Infer: 9.77 ms/batch
  Batch [80/5000] Top-1: 88.00% | Top-5: 100.00% | Infer: 9.80 ms/batch
  Batch [90/5000] Top-1: 90.00% | Top-5: 100.00% | Infer: 9.69 ms/batch
  Batch [100/5000] Top-1: 90.00% | Top-5: 100.00% | Infer: 9.68 ms/batch
  Batch [110/5000] Top-1: 90.00% | Top-5: 100.00% | Infer: 9.60 ms/batch
  Batch [120/5000] Top-1: 88.89% | Top-5: 98.89% | Infer: 9.56 ms/batch
  Batch [130/5000] Top-1: 90.00% | Top-5: 99.00% | Infer: 9.36 ms/batch
  Batch [140/5000

/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of maxpool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(
/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(


[data] Filtered to 5000 samples across 100 classes.
Evaluating on 5000 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/5000] Top-1: 80.00% | Top-5: 100.00% | Infer: 10.72 ms/batch
  Batch [50/5000] Top-1: 90.00% | Top-5: 100.00% | Infer: 10.46 ms/batch
  Batch [60/5000] Top-1: 90.00% | Top-5: 100.00% | Infer: 10.32 ms/batch
  Batch [70/5000] Top-1: 92.50% | Top-5: 100.00% | Infer: 10.26 ms/batch
  Batch [80/5000] Top-1: 92.00% | Top-5: 98.00% | Infer: 9.82 ms/batch
  Batch [90/5000] Top-1: 93.33% | Top-5: 98.33% | Infer: 9.86 ms/batch
  Batch [100/5000] Top-1: 94.29% | Top-5: 98.57% | Infer: 9.93 ms/batch
  Batch [110/5000] Top-1: 93.75% | Top-5: 98.75% | Infer: 9.88 ms/batch
  Batch [120/5000] Top-1: 92.22% | Top-5: 97.78% | Infer: 9.80 ms/batch
  Batch [130/5000] Top-1: 93.00% | Top-5: 98.00% | Infer: 9.75 ms/batch
  Batch [140/5000] Top-1: 92.73% | Top-5: 98.18% | Infer: 9.82 ms/batch
  Batch [150/5000] Top-1: 90.83% |

## Per-Run Results

In [7]:
rows = []
for rec in all_records:
    r = rec["results"]
    extra = rec.get("config_extra", rec.get("config", {}))
    bits = extra.get("input_quant_bits", rec["config"].get("input_quant_bits", None))
    prec = extra.get("qat_precision", "int8")
    seed = extra.get("seed", 42)
    rows.append({
        "qat_precision": f"qat_{prec}",
        "input_bits": bits,
        "seed": seed,
        "top1": r["top1_acc"],
        "top5": r["top5_acc"],
        "lat_ms": r["infer_ms_avg"],
        "throughput": r.get("throughput_infer_sps", None),
    })

df_all = pd.DataFrame(rows).sort_values(
    ["qat_precision", "input_bits", "seed"], ascending=[True, True, True]
).reset_index(drop=True)
df_all

,qat_precision,input_bits,seed,top1,top5,lat_ms,throughput
0,qat_int4,1,1,69.175,89.115,10.097,99.035
1,qat_int4,1,2,69.618,88.893,10.641,93.976
2,qat_int4,1,42,68.873,89.215,10.017,99.831
3,qat_int4,2,1,76.278,92.535,10.236,97.698
4,qat_int4,2,2,76.157,93.159,10.311,96.985
5,qat_int4,2,42,75.915,92.455,9.993,100.073
6,qat_int4,4,1,77.887,93.280,11.201,89.276
7,qat_int4,4,2,77.545,93.501,10.408,96.077
8,qat_int4,4,42,77.445,93.119,9.394,106.446
9,qat_int4,8,1,77.223,93.320,10.195,98.083


## Averaged Results (mean ± std across seeds)

In [8]:
avg_df = df_all.groupby(["qat_precision", "input_bits"]).agg(
    top1_mean=("top1", "mean"),
    top1_std=("top1", "std"),
    top5_mean=("top5", "mean"),
    top5_std=("top5", "std"),
    infer_ms_mean=("lat_ms", "mean"),
    infer_ms_std=("lat_ms", "std"),
    throughput_mean=("throughput", "mean"),
    n_seeds=("seed", "count"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["qat_precision", "input_bits"], ascending=[True, True]
).reset_index(drop=True)
avg_df

,qat_precision,input_bits,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean,n_seeds
0,qat_int4,1,69.222,0.374,89.074,0.165,10.252,0.339,97.614,3
1,qat_int4,2,76.117,0.184,92.716,0.385,10.180,0.166,98.252,3
2,qat_int4,4,77.626,0.232,93.300,0.192,10.335,0.906,97.266,3
3,qat_int4,8,77.257,0.173,93.219,0.112,10.022,0.558,99.995,3
4,qat_int8,1,69.349,0.469,89.457,0.073,8.778,0.605,114.292,3
5,qat_int8,2,76.714,0.338,93.099,0.232,8.938,0.811,112.517,3
6,qat_int8,4,78.417,0.316,93.474,0.141,8.732,0.457,114.732,3
7,qat_int8,8,77.995,0.141,93.374,0.091,8.738,0.290,114.524,3


In [9]:
out_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "qat_torch_avg_test.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")

Saved to /home/pf4636/code/resnet/quantized_resnets/resultsv2/test_runs/qat_torch_avg_test.json
